# Amparo -- M3 · S08: RAG avanzado

Monta las dos tecnicas de recuperacion avanzada (hybrid search y reranking)
sobre el RAG base. La evaluacion con juez (RAGAS) esta en
`m3_s10_rag_agentico.ipynb`. Las rutas agenticas de la S10 (tool use y ReAct) se
retiraron el 2026-10-08 (C11, `docs/m3_decisiones_rag.md` seccion 28).
Toda la logica vive en `tools/rag/`; aca se clona el repo, se instala el stack
pesado y se corren las fases.

**Las tres configuraciones que se comparan** (lo unico que cambia entre ellas es
el retrieval; mismo prompt, mismo generador, mismo eval set):

| Config | `use_hybrid` | `use_rerank` | Retrieval |
|---|---|---|---|
| **A** | False | False | denso puro (coseno e5) |
| **B** | True | False | denso + BM25, fusion RRF |
| **C** | True | True | hybrid -> cross-encoder reordena |

**Requiere GPU** (T4 o superior): e5, el cross-encoder y Qwen2.5-7B corren aca.

Las decisiones de diseno de cada tecnica estan en `docs/m3_decisiones_rag.md`.

Cada respuesta se emite en formato Ragas (`pipeline.to_eval_record`) con los
flags `used_hybrid`/`used_rerank`, que identifican la configuracion que la
genero.

In [1]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "m3.5"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

Cloning into 'Amparo'...
remote: Enumerating objects: 1524, done.
remote: Counting objects: 100% (437/437), done.
remote: Compressing objects: 100% (321/321), done.
remote: Total 1524 (delta 182), reused 258 (delta 113), pack-reused 1087 (from 1)
Receiving objects: 100% (1524/1524), 12.01 MiB | 30.21 MiB/s, done.
Resolving deltas: 100% (943/943), done.
/content/Amparo


In [2]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/verificacion.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG avanzado "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

Repo OK.


In [3]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.4 MB/s eta 0:00:00


In [4]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 138.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.2 MB/s eta 0:00:00


## Fase 1 -- Cargar el indice y construir el BM25

El indice denso (FAISS) se construye en `m3_s07_rag_ingenuo.ipynb` y se guarda en Drive.
Aca se carga ya hecho. El indice BM25 se construye en memoria a partir de la
metadata del store (los mismos chunks, en el mismo orden): es Python puro y
tarda segundos; se construye una vez y se reusa en toda la corrida.

In [5]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

Mounted at /content/drive
Indice copiado de Drive a artifacts/.


In [6]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

Indice denso: 11975 chunks | BM25: 11975 chunks


## Fase 2 -- Comparacion lado a lado: A vs B vs C sobre el retrieval

Antes de generar, se mira solo lo que recupera cada configuracion. Tres consultas
elegidas para exponer donde cada tecnica aporta:

1. **Termino exacto** (`articulo 64` del CST): donde el denso difumina y BM25
   rescata el chunk correcto -> B/C deberian subirlo.
2. **Coloquial** (lenguaje del usuario, sin terminos tecnicos): donde el denso ya
   es fuerte -> hybrid no debe empeorarlo.
3. **Fuera del corpus**: la valvula de escape debe activarse en las tres.

Solo se recupera, sin generar: es el diagnostico por consulta, mas informativo
que la metrica agregada.

In [7]:
from tools.rag.retrieve import retrieve

SISTEMAS = {
    "A denso": dict(use_hybrid=False, use_rerank=False),
    "B +hybrid": dict(use_hybrid=True, use_rerank=False),
    "C +rerank": dict(use_hybrid=True, use_rerank=True),
}

consultas_demo = [
    "que dice el articulo 64 del codigo sustantivo del trabajo",   # termino exacto
    "me despidieron sin pagarme la liquidacion, que hago",          # coloquial
    "cual es el plazo para apelar una multa de transito en Argentina",  # fuera de corpus
]

for q in consultas_demo:
    print("=" * 78)
    print("CONSULTA:", q)
    for nombre, banderas in SISTEMAS.items():
        # min_score=None para ver que recupera cada configuracion antes de la
        # valvula de escape.
        crudos = retrieve(q, store, top_k=3, min_score=None, bm25=bm25, **banderas)
        citas = [f"{r.cita} (dense={r.dense_score:.2f})" if r.dense_score is not None else f"{r.cita} (solo BM25)" for r in crudos]
        print(f"  {nombre}: {citas}")

CONSULTA: que dice el articulo 64 del codigo sustantivo del trabajo


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  C +rerank: ['Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 65 (solo BM25)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 5 (dense=0.88)', 'Ley 1010 de 2006 (Ley de Acoso Laboral), Articulo 10 (solo BM25)']
CONSULTA: me despidieron sin pagarme la liquidacion, que hago
  A denso: ['Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 402 (dense=0.83)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 53 (dense=0.83)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 403 (dense=0.83)']
  B +hybrid: ['Ley 2452 de 2025 (Codigo Procesal del Trabajo y de la Seguridad Social), Articulo 326 (dense=0.82)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 402 (dense=0.83)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 380 (dense=0.83)']
  C +rerank: ['Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 267 (dense=0.83)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), 

## Fase 3 -- Cargar el generador

Qwen2.5-7B-Instruct, cargado una sola vez y reusado. `USE_LORA=True` genera con
el adaptador LoRA encima.

In [8]:
# Amparo = modelo fine-tuneado de M1 (LoRA) + RAG: es el sistema que se entrega,
# asi que la corrida principal usa el adaptador. False sirve como ablacion
# ("¿el fine-tuning sigue aportando con RAG?"); OJO: en S08 los archivos de
# corridas_abc/ no llevan sufijo; una segunda corrida sobrescribe la primera.
USE_LORA = True
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo cargado.


## Fase 4 -- Corrida A/B/C sobre el eval set (con latencia)

Corre las tres configuraciones sobre `data/eval_set.json` y produce, por cada
una, la lista de registros en formato Ragas (`pipeline.to_eval_record`). Mide la
latencia por consulta de cada configuracion: toda tecnica cobra en tiempo, y ese
costo se reporta junto a la calidad.

Las tres corridas quedan etiquetadas por configuracion (`used_hybrid`/
`used_rerank`) y son comparables: mismo eval set, mismo generador.

In [9]:
import json, time
from tools.evaluation import eval_set as eval_set_mod

registros = eval_set_mod.load_eval_set()
print(f"{len(registros)} registros ({len(eval_set_mod.gold_examples(registros))} gold, "
      f"{len(eval_set_mod.adversarial_examples(registros))} adversariales)")

corridas = {}   # nombre_config -> lista de eval_records
latencias = {}  # nombre_config -> seg/consulta

for nombre, banderas in SISTEMAS.items():
    print(f"\nCorriendo configuracion {nombre} ...")
    records, t0 = [], time.perf_counter()
    for reg in registros:
        consulta = reg["messages"][1]["content"]
        resultado = pipeline.answer_query(
            consulta, store, use_lora=USE_LORA, bm25=bm25,
            model_bundle=model_bundle, **banderas,
        )
        records.append(pipeline.to_eval_record(resultado, reg))
    latencias[nombre] = (time.perf_counter() - t0) / len(registros)
    corridas[nombre] = records
    print(f"  {nombre}: {latencias[nombre]:.2f} seg/consulta")

75 registros (45 gold, 30 adversariales)

Corriendo configuracion A denso ...
  A denso: 7.33 seg/consulta

Corriendo configuracion B +hybrid ...
  B +hybrid: 7.22 seg/consulta

Corriendo configuracion C +rerank ...
  C +rerank: 7.38 seg/consulta


In [10]:
# Persistir las tres corridas en Drive para levantarlas sin re-generar.
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_abc"
destino.mkdir(parents=True, exist_ok=True)

for nombre, records in corridas.items():
    slug = nombre.split()[0].lower()   # a / b / c
    ruta = destino / f"eval_records_config_{slug}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"  {nombre} -> {ruta}")

print("\nLatencia por consulta:")
for nombre, seg in latencias.items():
    print(f"  {nombre}: {seg:.2f} s")

# La latencia es un resultado, no solo algo que se mira: es la mitad de la
# decision de si hybrid o rerank se justifican. Si solo se imprime, se pierde
# al cerrar el notebook y la carpeta de results queda sin ella.
with open(destino / "latencias_abc.json", "w", encoding="utf-8") as f:
    json.dump(latencias, f, ensure_ascii=False, indent=2)
print("\nLatencias ->", destino / "latencias_abc.json")

  A denso -> /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/eval_records_config_a.json
  B +hybrid -> /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/eval_records_config_b.json
  C +rerank -> /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/eval_records_config_c.json

Latencia por consulta:
  A denso: 7.33 s
  B +hybrid: 7.22 s
  C +rerank: 7.38 s

Latencias -> /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/latencias_abc.json


## Fase 5 -- ¿Que busqueda es mejor?

Esta es la pregunta de la S08. Amparo solo puede responder bien si **primero
encuentra la norma correcta**: si la busqueda trae el articulo equivocado (o
nada), el modelo responde mal o inventa, por bueno que sea el prompt. Por eso
aqui se deja fijo todo lo demas (mismo prompt, mismo modelo, mismas preguntas) y
lo **unico que cambia es como se busca**:

| Config | Como busca | En palabras simples |
|---|---|---|
| **A** denso | embeddings e5 (coseno) | busca por **significado**: entiende "me echaron del trabajo" aunque la norma diga "terminacion del contrato", pero confunde numeros ("articulo 64" con el 46 o el 65) |
| **B** + hybrid | denso + BM25, fusion RRF | suma una busqueda por **palabras exactas**: rescata "articulo 64", "Ley 1480", "habeas data", que el denso difumina |
| **C** + rerank | hybrid y un cross-encoder reordena | trae mas candidatos y un segundo modelo **lee cada par pregunta-articulo** para dejar arriba los que de verdad responden y quitar el ruido |

**Que significa "mejor".** No hay una sola nota; cada numero de la tabla mira
una cosa, y todos se calculan sin juez, directo de las respuestas guardadas:

- **Consultas sin contexto:** cuantas veces la busqueda no trajo nada. Menos es
  mejor en los casos gold (en los adversariales, a veces no traer nada es lo
  correcto).
- **Honestidad con las fuentes:** si la respuesta cita solo articulos que la
  busqueda trajo, y la **prudencia** en los adversariales (sin juez, reglas del
  repo).
- **Latencia:** cada tecnica cuesta tiempo; una mejora que no paga su costo no se
  justifica.

La calidad de las respuestas con juez (RAGAS: si la busqueda trae lo necesario y
sin ruido) se mide en `m3_s10_rag_agentico.ipynb`, que es donde esta el juez Groq.

**Antes y despues.** La primera corrida (2026-09-27) destapo tres fallas: la
valvula de escape del prompt nunca se activo, hubo citas inventadas, y el hybrid
descartaba justo lo que BM25 encontraba (por eso B y C traian menos articulos que
A). Se corrigieron en el codigo (`docs/m3_decisiones_rag.md`, seccion 25). La
tabla de abajo compara esa corrida (`corridas_abc_antes/`) con la corrida con las
correcciones (`corridas_abc/`).

### Tabla de comparacion A/B/C

**Sin juez** (se calcula de los archivos `eval_records_config_{a,b,c}.json`, con
`ragas_metrics.tasas_de_escape` y `dspy_prompt.puntaje_de_record`):

| | A denso | B +hybrid | C +rerank |
|---|---|---|---|
| Consultas sin contexto (gold) | 8 (5) | 8 (5) | 8 (5) |
| Articulos por consulta | 3.87 | 3.81 | 3.73 |
| Respuestas gold que citan algun articulo | 2/45 (4 %) | 1/45 (2 %) | 2/45 (4 %) |
| Respuestas gold con cita no respaldada | 0 | 0 | 0 |
| Respuestas corregidas por la verificacion de citas | 0 | 0 | 0 |
| Prudencia en adversariales | 30/30 | 30/30 | 30/30 |
| Honestidad con las fuentes (0-1) | 0.747 | 0.740 | 0.747 |
| Uso de la frase de escape | 8/75 (todos por codigo) | 8/75 (todos por codigo) | 9/75 (8 por codigo) |
| Latencia (s/consulta) | 5.47 | 5.42 | 5.57 |

Corrida del **2026-10-07**, commit `d7de3c0a`, `USE_LORA = True`, indice de 3420
chunks (`4269a04083d935fd`), eval set de 75 casos (`a5151999c2095d00`), A100 80GB.
El manifiesto completo esta en `results/m3_s08_2026-10-07/run_manifest.json`.

**Por que ya no hay columnas "antes".** La corrida del 2026-09-27 midio sobre un
eval set de 56 casos (50 gold + 6 adversariales); esta mide sobre 75 (45 gold +
30 adversariales). Son poblaciones distintas, asi que poner las dos en la misma
tabla compara cosas que no se pueden comparar -- el mismo error que la fase 5c
tenia antes de la seccion 25. Las cifras del 27-09 siguen en
`results/m3_s08_busqueda_2026-09-27.md`, con su eval set anotado.

**Que dicen los numeros:**

- **Las tres busquedas traen la misma cantidad y se quedan sin contexto en los
  mismos 8 casos** (9052, 9058, 9061, 9065, 9066, 9101, 9108, 9110), con 4.04
  chunks por consulta en las tres. Pero **recuperan cosas distintas**: A y B
  difieren en los contextos de 49 de las 75 preguntas. Es decir, el piso de 0.82
  y el `TOP_K = 5` dejan pasar el mismo *numero* de chunks en las tres
  configuraciones, y lo que cambia es *cuales*. Por eso ninguna metrica de
  conteo las separa, y la diferencia de calidad la tiene que decidir RAGAS.
- **5 casos gold se quedan sin contexto.** Son consultas que deberian encontrar
  norma y no encuentran nada por encima del piso. Es el numero a atacar: ni
  hybrid ni rerank lo mueven, porque los tres comparten el mismo filtro.
- **El sistema casi dejo de citar articulos**: 1 o 2 de 45 respuestas gold citan
  alguno. La honestidad sube a ~0.74 (contra 0.643 el 27-09), pero sube en parte
  *porque* no citar no puede ser una cita no respaldada. Conviene leerla junto
  con la fila de arriba, no sola.
- **Cero citas inventadas y cero correcciones**: la verificacion de citas no tuvo
  que rechazar nada. Con tan pocas citas, esta fila no dice mucho todavia.
- **La prudencia en adversariales es perfecta** (30/30) en las tres, ahora con 30
  casos y no 6.
- **La latencia no cambia** (~5.4-5.6 s): la domina la generacion, no la busqueda.
  Hybrid y rerank salen practicamente gratis.

**Con juez (RAGAS, fase 5b de `m3_s10_rag_agentico.ipynb`, juez Groq
`openai/gpt-oss-120b`, sobre los 45 casos gold):**

| Config | Context recall | Context precision | Faithfulness | Answer relevancy |
|---|---|---|---|---|
| **A denso** | **0.39** | 0.56 | **0.42** | 0.76 |
| B + hybrid | 0.37 | **0.59** | 0.39 | 0.76 |
| C + rerank | 0.34 | 0.58 | 0.38 | 0.76 |

**Gana A, el denso puro.** Es el mejor en context recall y faithfulness; B gana
por poco en precision y C queda ultimo en las dos que importan. Ni hybrid ni
rerank pagan su costo, y la latencia es la misma en las tres (5.4-5.6 s).

**Esto invierte la conclusion del 2026-09-27**, que daba C como mejor (recall
0.48). El informe de entonces no se equivoco: media sobre otro eval set, mas
facil. La razon del cambio esta abajo.

**El reranker se rompe en preguntas compuestas.** Separando los 45 gold entre
las 26 preguntas que venian del eval set anterior y las 19 que lo reemplazaron
el 2026-10-06 (commit `4d68d86`, mas compuestas a proposito):

| Config | recall en las 26 viejas | recall en las 19 nuevas |
|---|---|---|
| A denso | 0.50 | 0.25 |
| B + hybrid | 0.49 | 0.20 |
| C + rerank | 0.50 | **0.11** |

En las viejas las tres empatan y C es el mejor en precision (0.67). En las
nuevas C se desploma. Tiene explicacion mecanica: el cross-encoder puntua pares
(pregunta, chunk) y favorece el que calza con el aspecto dominante, desplazando
al que cubre el segundo. Una pregunta que exige combinar dos cosas pierde una.

**Pero el cuello de botella no es ninguna de las tres tecnicas.** En 19 de los
45 casos gold (42 %) el context recall es 0: la busqueda no trajo NADA util. Y
de esos, 11 preguntan por categorias que el corpus no indexa -- cubre 9
(`corpus.CATEGORIAS_OBJETIVO`) y el eval set pregunta por 27. Partido asi:

| | casos gold | recall 0 | recall promedio |
|---|---|---|---|
| Categorias del corpus | 24 | 8 (33 %) | 0.49 |
| Fuera del corpus | 21 | 11 (52 %) | 0.29 |

No es un fallo de ranking: las normas no estan. `codigo_comercio_decreto_410_1971.md`
esta descargado en el repo pero marcado `FUERA_DE_ALCANCE`.

Y el efecto se arrastra a la generacion: la faithfulness es 0.58 cuando el
contexto sirve y 0.13 cuando no trajo nada. El 0.42 agregado lo produce ese 42 %,
no un modelo peor -- 0.58 esta en el mismo rango que el 27-09 (0.55-0.61).

---
### Contenido

1. Codigo de las dos tecnicas (`tools/rag/hybrid.py`, `tools/rag/rerank.py`),
   integradas en `retrieve.py`/`pipeline.py` por bandera, con tests que corren
   sin GPU (`tests/rag/`).
2. Las tres corridas A/B/C en formato Ragas y la latencia por configuracion
   (Fase 4).
3. Las decisiones de diseno en `docs/m3_decisiones_rag.md`.

*Amparo · M3 · S08 · RAG avanzado.*

## Manifiesto de la corrida

Guarda junto a los resultados **con que** se produjeron: commit, versiones de
las librerias y huellas del indice, el corpus, el dataset, el eval set y el
adaptador.

Huellas y no fechas: dos corridas pueden decir las dos "2026-10-07" y haber
medido sobre indices distintos. Si declaran el mismo hash, midieron sobre lo
mismo.


In [11]:
# Manifiesto de la corrida: commit, versiones y huellas del indice, el corpus,
# el dataset, el eval set y el adaptador. Sin esto no se sabe con que salio cada
# cifra -- el adaptador de Drive ya se sobrescribio una vez (6 de octubre) y las
# corridas anteriores a esa fecha quedaron sin forma de reproducirse.
#
# Lee las variables del notebook con globals().get(): un manifiesto describe la
# corrida, no puede tumbarla. Si falta alguna, queda en None y se anota.
import pathlib
import subprocess

from tools.rag import manifiesto

_hardware = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()

_store = globals().get("store")
_registros = globals().get("registros")

_manifiesto = manifiesto.construir(
    "s08",
    use_lora=bool(globals().get("USE_LORA", False)),
    n_chunks=len(_store.metadata) if _store is not None else None,
    n_eval_set=len(_registros) if _registros is not None else None,
    hardware=_hardware,
)
_destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_abc"
print("Manifiesto:", _manifiesto.guardar(_destino))
print(f"  commit {_manifiesto.git_commit[:8]} | indice {_manifiesto.hash_indice} "
      f"| adaptador {_manifiesto.hash_adaptador} | eval set {_manifiesto.hash_eval_set}")
print(f"  chunks {_manifiesto.n_chunks} | casos {_manifiesto.n_eval_set}")


Manifiesto: /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/run_manifest.json
  commit 8fd29a69 | indice 19657d22583f93d0 | adaptador 8ce3cc2bc9306974 | eval set a5151999c2095d00
  chunks 11975 | casos 75
